# Entrena a tu propio David

Ajusta **Gemma 4 E2B** con LoRA para que convierta mensajes de WhatsApp de una ferretería en tickets JSON.
Sigue el codelab paso a paso: cada sección de este notebook corresponde a un paso.

- Entorno: Colab con GPU **T4** (menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*).
- No uses *Ejecutar todo*: corre las celdas en orden, una por una, mientras lees el paso.
- Código y datos: [https://github.com/ykro/codelabs-2027/tree/main/entrena-tu-david](https://github.com/ykro/codelabs-2027/tree/main/entrena-tu-david)

## Paso 3 · Abre el notebook e instala Unsloth

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
%%capture
!pip install --no-deps unsloth==2026.10.4 unsloth_zoo==2026.10.4 bitsandbytes==0.50.2 trl==1.15.0 cut_cross_entropy==25.1.1
!pip install sentencepiece protobuf "datasets==4.8.5" "huggingface_hub>=1.5.0,<2.0" hf_transfer gradio==6.26.0
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0" "torchao>=0.16.0"

In [3]:
import unsloth, transformers, trl, peft, gradio
print("unsloth", unsloth.__version__, "· transformers", transformers.__version__,
      "· trl", trl.__version__, "· peft", peft.__version__, "· gradio", gradio.__version__)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!


unsloth 2026.10.4 · transformers 5.5.0 · trl 1.15.0 · peft 0.20.0 · gradio 6.26.0


## Paso 4 · Carga Gemma 4 E2B

In [4]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-4-E2B-it",
    max_seq_length=1024,
    load_in_4bit=False,
)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

==((====))==  Unsloth 2026.10.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Using float16 precision for gemma4 won't work! Using float32.


Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

In [5]:
import torch

gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} · memoria total: {gpu.total_memory / 1024**3:.1f} GB")
print(f"Memoria reservada después de cargar: {torch.cuda.max_memory_reserved() / 1024**3:.1f} GB")

GPU: Tesla T4 · memoria total: 14.6 GB
Memoria reservada después de cargar: 10.3 GB


## Paso 5 · Mira el dataset y escribe el prompt

In [6]:
import json, os, urllib.request

URL = "https://raw.githubusercontent.com/ykro/codelabs-2027/main/entrena-tu-david/datos"
for nombre in ["train.jsonl", "test.jsonl"]:
    if not os.path.exists(nombre):  # si subiste el archivo a Colab, usa el tuyo
        urllib.request.urlretrieve(f"{URL}/{nombre}", nombre)

train = [json.loads(linea) for linea in open("train.jsonl", encoding="utf-8")]
test = [json.loads(linea) for linea in open("test.jsonl", encoding="utf-8")]
print(f"train: {len(train)} ejemplos · test: {len(test)} ejemplos")

train: 1000 ejemplos · test: 150 ejemplos


In [7]:
import pandas as pd

pd.set_option("display.max_colwidth", 90)
pd.DataFrame([{"mensaje": r["mensaje"], **r["ticket"]} for r in train[:5]], dtype=object)

,mensaje,tipo,categoria,producto,cantidad,monto_q,urgencia
0,"Buenas tardes, mire que chilero su servicio, tráigame 15 palas para el sabado por favo...",pedido,herramientas,pala,15,None,media
1,"Fijese que ya les pague los Q 25 por los cinco blocks que pedi para hoy, puchis, queri...",seguimiento,obra_gris,block,5,25,alta
2,"¡Puchis, se me olvidaba! Necesito que me manden 20 llaves de paso esta semana, por fav...",pedido,plomeria,llave de paso,20,None,media
3,"Buenas. Fíjese que necesito pedirle 24 de arena, pero sería para mas adelante cuando u...",pedido,obra_gris,arena de río,24,None,baja
4,"Buenas, mándenme cinco clavos de tres para la obra, por favor, los necesitamos ahorita...",pedido,obra_gris,clavo de 3 pulgadas,5,None,alta


In [8]:
print(json.dumps(train[0]["origen"], ensure_ascii=False, indent=1))

{
 "tipo": "sintetico",
 "modelo": "gemini-3.8-flash",
 "servicio": "Vertex AI",
 "fecha": "2026-10-13",
 "semilla": 1821,
 "solicitud": "train-1821-00001",
 "pistas": {
  "producto_como": [
   "pala",
   "palas"
  ],
  "cantidad_como": "15",
  "monto_como": null,
  "urgencia_como": "para el sabado",
  "estilo": [
   "usa «patojo»",
   "usa «chilero»"
  ]
 },
 "validacion": "reglas de generar_datos.py: ok"
}


In [9]:
PROMPT = """Eres el asistente de Ferretería La Ceiba, en Guatemala. Convierte el mensaje del cliente en un ticket JSON con estos campos:
- tipo: pedido, cotizacion, reclamo o seguimiento
- categoria: obra_gris, techos, plomeria, electricidad, pintura, herramientas o jardin
- producto: el nombre del producto como lo vende la tienda, en singular
- cantidad: número entero, o null si no la dice
- monto_q: monto en quetzales, o null si no lo dice
- urgencia: alta, media o baja
Responde solo con el JSON.

Mensaje: {mensaje}"""


def conversacion(mensaje, ticket=None):
    turnos = [{"role": "user", "content": [{"type": "text", "text": PROMPT.format(mensaje=mensaje)}]}]
    if ticket is not None:
        respuesta = json.dumps(ticket, ensure_ascii=False)
        turnos.append({"role": "assistant", "content": [{"type": "text", "text": respuesta}]})
    return turnos

## Paso 6 · Mide el modelo base

In [10]:
import re, unicodedata

CAMPOS = ["tipo", "categoria", "producto", "cantidad", "monto_q", "urgencia"]
PERMITIDOS = {
    "tipo": ["pedido", "cotizacion", "reclamo", "seguimiento"],
    "categoria": ["obra_gris", "techos", "plomeria", "electricidad", "pintura", "herramientas", "jardin"],
    "urgencia": ["alta", "media", "baja"],
}


def normalizar(texto):
    t = unicodedata.normalize("NFKD", str(texto).lower())
    return " ".join("".join(c for c in t if not unicodedata.combining(c)).split())


def leer_ticket(respuesta):
    """El ticket que escribió el modelo, o None si no es un JSON válido con los 6 campos."""
    texto = re.sub(r"^```(?:json)?|```$", "", respuesta.strip()).strip()
    try:
        ticket = json.loads(texto)
    except json.JSONDecodeError:
        return None
    if not isinstance(ticket, dict) or set(ticket) != set(CAMPOS):
        return None
    if any(ticket[c] not in valores for c, valores in PERMITIDOS.items()):
        return None
    return ticket


def igual(campo, a, b):
    if campo in ("cantidad", "monto_q"):
        try:
            return (a is None and b is None) or abs(float(a) - float(b)) < 0.01
        except (TypeError, ValueError):
            return False
    return normalizar(a) == normalizar(b)


def calificar(respuestas, filas):
    validos, perfectos, aciertos = 0, 0, {c: 0 for c in CAMPOS}
    for respuesta, fila in zip(respuestas, filas):
        ticket = leer_ticket(respuesta)
        if ticket is None:
            continue
        validos += 1
        bien = [igual(c, ticket[c], fila["ticket"][c]) for c in CAMPOS]
        for c, ok in zip(CAMPOS, bien):
            aciertos[c] += ok
        perfectos += all(bien)
    n = len(filas)
    return {"JSON válido": validos / n,
            "Campos correctos": sum(aciertos.values()) / (n * len(CAMPOS)),
            "Tickets perfectos": perfectos / n,
            **{f"· {c}": aciertos[c] / n for c in CAMPOS}}


def tabla(**columnas):
    return pd.DataFrame(columnas).map(lambda x: f"{x:.0%}")

In [11]:
import time

texto_tok = getattr(tokenizer, "tokenizer", tokenizer)


def responder(mensajes, lote=16):
    texto_tok.padding_side = "left"  # para generar, el relleno va a la izquierda
    salidas = []
    for i in range(0, len(mensajes), lote):
        prompts = [tokenizer.apply_chat_template(conversacion(m), tokenize=False, add_generation_prompt=True)
                   for m in mensajes[i:i + lote]]
        entrada = texto_tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        salida = model.generate(**entrada, max_new_tokens=96, do_sample=False)
        salidas += texto_tok.batch_decode(salida[:, entrada["input_ids"].shape[1]:], skip_special_tokens=True)
    return salidas

In [12]:
FastModel.for_inference(model)
inicio = time.time()
respuestas_base = responder([r["mensaje"] for r in test])
print(f"{len(test)} mensajes en {time.time() - inicio:.0f} s")
base = calificar(respuestas_base, test)
tabla(**{"Gemma sin ajustar": base})

150 mensajes en 140 s


,Gemma sin ajustar
JSON válido,96%
Campos correctos,76%
Tickets perfectos,23%
· tipo,84%
· categoria,61%
· producto,57%
· cantidad,89%
· monto_q,91%
· urgencia,73%


In [13]:
for fila, respuesta in list(zip(test, respuestas_base))[:3]:
    print("MENSAJE  ", fila["mensaje"])
    print("ESPERADO ", json.dumps(fila["ticket"], ensure_ascii=False))
    print("BASE     ", " ".join(respuesta.split()), "\n")

MENSAJE   Qué chilero ese chunche, quiero pedir un par de thinner para entrega porfa, tengo Q 140 para el otro mes.
ESPERADO  {"tipo": "pedido", "categoria": "pintura", "producto": "thinner", "cantidad": 2, "monto_q": 140, "urgencia": "baja"}
BASE      ```json { "tipo": "pedido", "categoria": "pintura", "producto": "Thinner", "cantidad": 2, "monto_q": 140, "urgencia": "media" } ``` 

MENSAJE   Fíjese que el patojo me cobró 805 pesos por 3 de piedrin pero vino de mala calidad, no me digan que lo solucionan mas adelante.
ESPERADO  {"tipo": "reclamo", "categoria": "obra_gris", "producto": "piedrín", "cantidad": 3, "monto_q": 805, "urgencia": "baja"}
BASE      ```json { "tipo": "reclamo", "categoria": "jardin", "producto": "Patojo", "cantidad": 3, "monto_q": 805, "urgencia": "alta" } ``` 

MENSAJE   Buenas tardes, fíjese que el patojo me cobró 195 varos por 5 llaves de paso pero vienen defectuosas, necesito que usted me las cambie hoy mismo por favor.
ESPERADO  {"tipo": "reclamo", "categor

## Paso 7 · Configura LoRA

In [14]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,     # solo texto: la parte de visión queda congelada
    finetune_language_layers=True,
    finetune_attention_modules=True,  # q, k, v, o
    finetune_mlp_modules=True,        # gate, up, down
    r=16,                             # tamaño de las matrices pequeñas que sí aprenden
    lora_alpha=16,                    # cuánto pesa lo aprendido (alpha / r = 1)
    lora_dropout=0,
    bias="none",
    random_state=3407,
)
model.print_trainable_parameters()

trainable params: 25,337,856 || all params: 5,148,515,872 || trainable%: 0.4921


## Paso 8 · Entrena

In [15]:
from datasets import Dataset

textos = [tokenizer.apply_chat_template(conversacion(r["mensaje"], r["ticket"]), tokenize=False).removeprefix("<bos>")
          for r in train]
datos = Dataset.from_dict({"text": textos})
print(textos[0])

<|turn>user
Eres el asistente de Ferretería La Ceiba, en Guatemala. Convierte el mensaje del cliente en un ticket JSON con estos campos:
- tipo: pedido, cotizacion, reclamo o seguimiento
- categoria: obra_gris, techos, plomeria, electricidad, pintura, herramientas o jardin
- producto: el nombre del producto como lo vende la tienda, en singular
- cantidad: número entero, o null si no la dice
- monto_q: monto en quetzales, o null si no lo dice
- urgencia: alta, media o baja
Responde solo con el JSON.

Mensaje: Buenas tardes, mire que chilero su servicio, tráigame 15 palas para el sabado por favor, ahí el patojo va a estar esperando para recibirlas.<turn|>
<|turn>model
{"tipo": "pedido", "categoria": "herramientas", "producto": "pala", "cantidad": 15, "monto_q": null, "urgencia": "media"}<turn|>



In [16]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=datos,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,   # 4 x 2 = 8 ejemplos por paso
        num_train_epochs=2,              # cada ejemplo se ve 2 veces
        learning_rate=2e-4,
        warmup_steps=10,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.001,
        logging_steps=25,
        seed=3407,
        output_dir="salidas",
        report_to="none",
    ),
)
trainer = train_on_responses_only(trainer)  # solo aprende de la respuesta (el JSON), no del prompt

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1000 [00:00<?, ? examples/s]

Unsloth: Auto-detected instruction_part = '<|turn>user\n' and response_part = '<|turn>model\n'


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [17]:
FastModel.for_training(model)
stats = trainer.train()
print(f"Entrenamiento: {stats.metrics['train_runtime'] / 60:.1f} min")
print(f"Pico de memoria de GPU: {torch.cuda.max_memory_reserved() / 1024**3:.1f} GB de {gpu.total_memory / 1024**3:.1f} GB")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 2 | Total steps = 250
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 25,337,856 of 5,148,515,872 (0.49% trained)


Step,Training Loss
25,0.333376
50,0.046468
75,0.017471
100,0.006471
125,0.009246
150,0.003545
175,0.004725
200,0.003508
225,0.001725
250,0.002005


Unsloth: Restored added_tokens_decoder metadata in salidas/checkpoint-250/tokenizer_config.json.


Entrenamiento: 9.4 min
Pico de memoria de GPU: 10.5 GB de 14.6 GB


## Paso 9 · Mide tu David

In [18]:
FastModel.for_inference(model)
respuestas_david = responder([r["mensaje"] for r in test])
david = calificar(respuestas_david, test)
tabla(**{"Gemma sin ajustar": base, "Tu David": david})

,Gemma sin ajustar,Tu David
JSON válido,96%,100%
Campos correctos,76%,99%
Tickets perfectos,23%,94%
· tipo,84%,99%
· categoria,61%,100%
· producto,57%,100%
· cantidad,89%,97%
· monto_q,91%,100%
· urgencia,73%,99%


In [19]:
cambios = [(f, b, d) for f, b, d in zip(test, respuestas_base, respuestas_david)
           if leer_ticket(d) and all(igual(c, leer_ticket(d)[c], f["ticket"][c]) for c in CAMPOS)
           and not (leer_ticket(b) and all(igual(c, leer_ticket(b)[c], f["ticket"][c]) for c in CAMPOS))]
print(f"{len(cambios)} mensajes que el modelo base fallaba y tu David resuelve. Tres de ellos:\n")
for fila, b, d in cambios[:3]:
    print("MENSAJE ", fila["mensaje"])
    print("BASE    ", " ".join(b.split()))
    print("DAVID   ", d.strip(), "\n")

107 mensajes que el modelo base fallaba y tu David resuelve. Tres de ellos:

MENSAJE  Qué chilero ese chunche, quiero pedir un par de thinner para entrega porfa, tengo Q 140 para el otro mes.
BASE     ```json { "tipo": "pedido", "categoria": "pintura", "producto": "Thinner", "cantidad": 2, "monto_q": 140, "urgencia": "media" } ```
DAVID    {"tipo": "pedido", "categoria": "pintura", "producto": "thinner", "cantidad": 2, "monto_q": 140, "urgencia": "baja"} 

MENSAJE  Fíjese que el patojo me cobró 805 pesos por 3 de piedrin pero vino de mala calidad, no me digan que lo solucionan mas adelante.
BASE     ```json { "tipo": "reclamo", "categoria": "jardin", "producto": "Patojo", "cantidad": 3, "monto_q": 805, "urgencia": "alta" } ```
DAVID    {"tipo": "reclamo", "categoria": "obra_gris", "producto": "piedrín", "cantidad": 3, "monto_q": 805, "urgencia": "baja"} 

MENSAJE  Buenas, por favor envíeme media docena de aspersores a mi dirección para el jueves. Quedo al pendiente de su confirmación, 

In [20]:
model.save_pretrained("david_lora")
_ = tokenizer.save_pretrained("david_lora")
!du -sh david_lora

Unsloth: Restored added_tokens_decoder metadata in david_lora/tokenizer_config.json.


128M	david_lora


### Opcional: compáralo con Gemini (Goliat)
Necesitas tu llave de AI Studio guardada en los *Secretos* de Colab con el nombre `GEMINI_API_KEY`. Sin llave, la celda solo avisa y puedes seguir.

In [ ]:
# Opcional: si no tienes llave de AI Studio, esta celda solo avisa y puedes seguir al paso 10.
from google import genai
from google.colab import userdata

MODELO_GEMINI = "gemini-3.8-flash"  # si llegas al límite del día: "gemini-3.5-flash-lite"
try:
    cliente = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
except Exception:
    cliente = None
    print("No encontré GEMINI_API_KEY en los secretos de Colab (o no le diste acceso). Salta este paso.")

if cliente:
    INSTRUCCIONES = (PROMPT.split("\n\nMensaje:")[0] + "\nVas a recibir varios mensajes numerados. "
                     "Responde con un arreglo JSON: un ticket por mensaje, en el mismo orden.")
    respuestas_gemini = []
    for i in range(0, len(test), 25):  # 25 mensajes por solicitud: 6 solicitudes en total
        grupo = test[i:i + 25]
        lista = "\n".join(f"{n}. {r['mensaje']}" for n, r in enumerate(grupo, 1))
        inicio = time.time()
        r = cliente.models.generate_content(model=MODELO_GEMINI, contents=f"{INSTRUCCIONES}\n\nMensajes:\n{lista}",
                                            config={"response_mime_type": "application/json", "temperature": 0})
        tickets = json.loads(r.text)[:len(grupo)]
        respuestas_gemini += [json.dumps(t, ensure_ascii=False) for t in tickets] + [""] * (len(grupo) - len(tickets))
        print(f"{len(respuestas_gemini)}/{len(test)} mensajes ({time.time() - inicio:.0f} s)")
    gemini = calificar(respuestas_gemini, test)
    display(tabla(**{"Gemma sin ajustar": base, "Tu David": david, "Gemini (Goliat)": gemini}))

25/150 mensajes (40 s)


50/150 mensajes (26 s)


75/150 mensajes (37 s)


100/150 mensajes (29 s)


125/150 mensajes (45 s)


150/150 mensajes (10 s)


,Gemma sin ajustar,Tu David,Gemini (Goliat)
JSON válido,96%,100%,100%
Campos correctos,76%,99%,91%
Tickets perfectos,23%,94%,52%
· tipo,84%,99%,98%
· categoria,61%,100%,93%
· producto,57%,100%,63%
· cantidad,89%,97%,95%
· monto_q,91%,100%,100%
· urgencia,73%,99%,94%


## Paso 10 · Pruébalo en una app

In [22]:
import gradio as gr
from html import escape

COLORES = {"alta": "#b42318", "media": "#b54708", "baja": "#067647"}


def tarjeta(respuesta):
    ticket = leer_ticket(respuesta)
    if ticket is None:
        return f'<div class="t mal"><b>No es un ticket válido</b><pre>{escape(respuesta.strip())}</pre></div>'
    filas = "".join(f"<dt>{c}</dt><dd>{escape('—' if ticket[c] is None else str(ticket[c]))}</dd>"
                    for c in CAMPOS if c != "urgencia")
    color = COLORES[ticket["urgencia"]]
    return (f'<div class="t"><span class="u" style="background:{color}">urgencia {ticket["urgencia"]}</span>'
            f"<dl>{filas}</dl></div>")


def crear_ticket(mensaje):
    FastModel.for_inference(model)
    david = responder([mensaje])[0]
    with model.disable_adapter():
        sin_ajustar = responder([mensaje])[0]
    return tarjeta(david), tarjeta(sin_ajustar)


CSS = """
.gradio-container {max-width: 760px !important; margin: auto; font-size: 16px}
.t {border: 1px solid #d0d5dd; border-radius: 12px; padding: 14px; background: #fff}
.t.mal {border-color: #fda29b; background: #fef3f2}
.t pre {white-space: pre-wrap; font-size: 14px}
.t dl {display: grid; grid-template-columns: auto 1fr; gap: 6px 12px; margin: 10px 0 0}
.t dt {color: #475467} .t dd {margin: 0; font-weight: 600; overflow-wrap: anywhere}
.u {color: #fff; border-radius: 999px; padding: 3px 10px; font-size: 14px}
button {min-height: 48px}
"""
CLARO = "() => { document.body.classList.remove('dark'); }"
ejemplos = [f["mensaje"] for f, _, _ in cambios[:4]]

with gr.Blocks(title="Ferretería La Ceiba · tickets") as app:
    gr.Markdown("## Ferretería La Ceiba\nPega un mensaje de WhatsApp y tu modelo lo convierte en ticket.")
    mensaje = gr.Textbox(label="Mensaje del cliente", lines=3,
                         placeholder="buenas, ocupo 3 bolsas de cemento para hoy, la obra está parada")
    boton = gr.Button("Crear ticket", variant="primary")
    with gr.Row():
        with gr.Column():
            gr.Markdown("**Tu David (ajustado)**")
            salida_david = gr.HTML()
        with gr.Column():
            gr.Markdown("**Gemma 4 E2B sin ajustar**")
            salida_base = gr.HTML()
    gr.Examples(ejemplos, inputs=mensaje, label="Ejemplos del test")
    boton.click(crear_ticket, mensaje, [salida_david, salida_base])

app.launch(share=True, css=CSS, js=CLARO, theme=gr.themes.Default(font=["Inter", "system-ui", "sans-serif"]))

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()


* Running on public URL: https://da57110ddd787b7798.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Paso 11 · Llévalo a tu laptop con Ollama (opcional)
Exporta solo el adaptador a GGUF (unos 50 MB). El modelo completo no cabe en la RAM de Colab gratis al convertirlo.

In [1]:
!git -c advice.detachedHead=false clone -q --depth 1 --branch b11541 https://github.com/ggml-org/llama.cpp
!pip install -q --no-deps ./llama.cpp/gguf-py

  Installing build dependencies ... done
  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


In [2]:
!python llama.cpp/convert_lora_to_gguf.py david_lora --outfile david-lora.gguf --outtype f16 2>&1 | tail -n 2
!ls -lh david-lora.gguf

Writing: 100%|██████████| 50.7M/50.7M [00:00<00:00, 225Mbyte/s]
INFO:lora-to-gguf:Model successfully exported to david-lora.gguf


-rw-r--r-- 1 root root 49M Oct 13 09:11 david-lora.gguf
